# Forecasting Dataset — Stage 2 (Daraz only)

Continues from `recalculate_historical_data.py`, which already re-derived
`unit_value`/`unit_type`/`unit_price` and excluded commodity-filter,
missing-price, parsing-failure, and outlier rows from
`product_prices_recalculated.csv` — none of that is redone here.

**Only Daraz is used for forecasting.** Naheed is loaded and diagnosed in
section 3, then dropped: its catalog is small and barely changes, it has
no rating/review data at all, and cross-store price matching only ever
covered a small fraction of its own sessions. The numbers backing that call
are printed in section 3, not just asserted.

Output: `forecasting_daraz_clean.csv` (session-level) and
`forecasting_daraz_daily_clean.csv` (daily — the actual modeling grain,
per the day/night investigation in sections 6-7).

## 1. Imports & config

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt

from utils.commodity_filters import is_valid_commodity 

pd.set_option('display.max_columns', 60)

CROSS_SOURCE_CATEGORIES = {'oil', 'ghee', 'rice', 'flour'}
SESSION_GAP_THRESHOLD  = pd.Timedelta('3h')   # items closer together than this belong to the same scrape run
CROSS_SOURCE_TOLERANCE = pd.Timedelta('3h')   # max gap to call two stores' runs "the same session" (diagnostic only)

LAG_WINDOWS = [1, 2, 7, 14]
ROLLING_SHORT = 7
ROLLING_LONG = 14
SPIKE_WINDOW = 14
SPIKE_MIN_PERIODS = 5
SPIKE_STD_MULT = 2

OUTDIR = Path('output')
OUTDIR.mkdir(parents=True, exist_ok=True)


In [ ]:
products_raw = pd.read_csv('products_rows.csv')

products_raw['passes_filter'] = products_raw.apply(
    lambda row: is_valid_commodity(row['name'], row['category']),
    axis=1,
)

removed = products_raw[~products_raw['passes_filter']]
print(f"{len(removed)} / {len(products_raw)} products fail the updated commodity filter")
print(removed['category'].value_counts())

products = products_raw[products_raw['passes_filter']].drop(columns=['passes_filter']).reset_index(drop=True)

## 2. Load, merge, base prep (both stores)

In [ ]:
prices = pd.read_csv('product_prices_rows.csv')
prices = prices[prices['pipeline_version'] == '2.0.0'].reset_index(drop=True)
original_prices = prices.copy()

merged = prices.merge(
    products[['id', 'name', 'category', 'source', 'url', 'keyword']],
    left_on='product_id', right_on='id', how='left', suffixes=('', '_product')
)
orphaned = merged['id_product'].isna()
merged = merged[~orphaned].drop(columns=['id_product']).reset_index(drop=True)

missing_unit = merged['unit_value'].isna() | merged['unit_type'].isna() | merged['unit_price'].isna()
merged = merged[~missing_unit].reset_index(drop=True)

print(f"products.csv: {len(products)} | prices (2.0.0 only): {len(prices)}")
print(f"orphaned (no product match): {orphaned.sum()} | missing unit fields: {missing_unit.sum()}")
print(f"rows remaining: {len(merged)}")

In [ ]:
df = merged[[
    'scraped_at', 'run_id', 'name', 'category', 'source', 'url',
    'unit_price', 'unit_type', 'unit_value',
    'final_price', 'original_price', 'sale_price', 'rating', 'review_count',
]].copy()
df['scraped_at'] = pd.to_datetime(df['scraped_at'])

df['has_rating'] = df['rating'].notna()
df['has_review'] = df['review_count'].notna()
df['rating_clean'] = df['rating'].fillna(0)
df['review_count_clean'] = df['review_count'].fillna(0)

df['is_discounted'] = df['sale_price'].notna() & df['original_price'].notna() & (df['sale_price'] < df['original_price'])
df['discount_pct'] = np.nan
m = df['is_discounted']
df.loc[m, 'discount_pct'] = ((df.loc[m, 'original_price'] - df.loc[m, 'sale_price']) / df.loc[m, 'original_price'] * 100).round(2)

df['hour']        = df['scraped_at'].dt.hour
df['day_of_week'] = df['scraped_at'].dt.dayofweek
df['week']        = df['scraped_at'].dt.isocalendar().week.astype(int)
df['month']       = df['scraped_at'].dt.month
df['is_weekend']  = df['day_of_week'].isin([5, 6])

print(df['source'].value_counts())


## 3. Why Daraz-only

Three checks, run on both stores before committing to Daraz-only:
1. Catalog size and per-product history depth
2. Rating/review availability
3. Per-product price variation, and cross-store session match coverage

If Naheed's numbers looked comparable to Daraz's here, it'd be worth
keeping as its own dataset. They don't.

In [ ]:
def store_summary(d, name):
    obs = d.groupby('url').size()
    cv = d.groupby('url')['unit_price'].apply(lambda s: s.std() / s.mean() if s.mean() else np.nan)
    print(f"[{name}] rows={len(d)}  products={d['url'].nunique()}  "
          f"obs/product min/median/max={obs.min()}/{obs.median():.0f}/{obs.max()}")
    print(f"[{name}] has_rating={d['has_rating'].mean()*100:.1f}%  has_review={d['has_review'].mean()*100:.1f}%")
    print(f"[{name}] per-product price coefficient of variation: median={cv.median():.4f}, "
          f"{(cv == 0).mean()*100:.1f}% of products show zero price variation across all observations")
    print()

for src, grp in df.groupby('source'):
    store_summary(grp, src)


In [ ]:
def assign_session_id(d):
    d = d.sort_values('scraped_at').copy()
    gaps = d['scraped_at'].diff()
    d['session_id'] = (gaps.isna() | (gaps > SESSION_GAP_THRESHOLD)).cumsum()
    return d

def session_medians(d, cat_set, label):
    scope = d[d['category'].isin(cat_set)]
    return (
        scope.groupby(['category', 'unit_type', 'session_id'])
        .agg(**{f'{label}_median': ('unit_price', 'median'), f'{label}_time': ('scraped_at', 'mean')})
        .reset_index().rename(columns={'session_id': f'{label}_session_id'})
    )

daraz_diag  = assign_session_id(df[df['source'] == 'daraz'])
naheed_diag = assign_session_id(df[df['source'] == 'naheed'])

daraz_med  = session_medians(daraz_diag, CROSS_SOURCE_CATEGORIES, 'daraz')
naheed_med = session_medians(naheed_diag, CROSS_SOURCE_CATEGORIES, 'naheed')

pairs = []
for (cat, ut), dg in daraz_med.groupby(['category', 'unit_type']):
    ng = naheed_med[(naheed_med['category'] == cat) & (naheed_med['unit_type'] == ut)]
    if ng.empty:
        continue
    matched = pd.merge_asof(
        dg.sort_values('daraz_time'),
        ng.drop(columns=['category', 'unit_type']).sort_values('naheed_time'),
        left_on='daraz_time', right_on='naheed_time',
        direction='nearest', tolerance=CROSS_SOURCE_TOLERANCE,
    )
    pairs.append(matched)

matched = pd.concat(pairs, ignore_index=True) if pairs else pd.DataFrame(columns=['naheed_median', 'naheed_session_id'])
matched_valid = matched[matched['naheed_median'].notna()] if len(matched) else matched
naheed_sessions_matched = matched_valid['naheed_session_id'].nunique() if len(matched_valid) else 0

print(f"Naheed sessions in cross-source categories (oil/ghee/rice/flour): {len(naheed_med)}")
if len(naheed_med):
    print(f"Naheed sessions matched by some Daraz session within {CROSS_SOURCE_TOLERANCE}: "
          f"{naheed_sessions_matched} ({naheed_sessions_matched / len(naheed_med) * 100:.1f}% coverage)")


## 4. Daraz-only from here

In [ ]:
daraz_df = df[df['source'] == 'daraz'].sort_values(['url', 'scraped_at']).reset_index(drop=True)
print(f"{len(daraz_df)} rows, {daraz_df['url'].nunique()} products")


## 5. Time-series feature engineering

In [ ]:
g = daraz_df.groupby('url', group_keys=False)

for lag in LAG_WINDOWS:
    daraz_df[f'price_lag_{lag}'] = g['unit_price'].shift(lag)

daraz_df['rolling_mean_7']  = g['unit_price'].transform(lambda s: s.rolling(ROLLING_SHORT, min_periods=1).mean())
daraz_df['rolling_std_7']   = g['unit_price'].transform(lambda s: s.rolling(ROLLING_SHORT, min_periods=1).std())
daraz_df['rolling_mean_14'] = g['unit_price'].transform(lambda s: s.rolling(ROLLING_LONG, min_periods=1).mean())
daraz_df['rolling_min_7']   = g['unit_price'].transform(lambda s: s.rolling(ROLLING_SHORT, min_periods=1).min())
daraz_df['rolling_max_7']   = g['unit_price'].transform(lambda s: s.rolling(ROLLING_SHORT, min_periods=1).max())

daraz_df['price_change']     = g['unit_price'].diff(1)
daraz_df['price_change_pct'] = g['unit_price'].transform(lambda s: s.pct_change(1) * 100)
daraz_df['price_accel']      = g['price_change'].transform(lambda s: s.diff(1))

daraz_df['discount_rate'] = daraz_df['discount_pct'] / 100
daraz_df['has_discount']  = daraz_df['is_discounted'].astype(int)
daraz_df['rolling_discount_rate'] = g['has_discount'].transform(lambda s: s.rolling(ROLLING_SHORT, min_periods=1).mean())

daraz_df['log_review_count'] = np.log1p(daraz_df['review_count_clean'])
daraz_df['rating_weighted']  = daraz_df['rating_clean'] * daraz_df['has_review'].astype(int)

rm = g['unit_price'].transform(lambda s: s.rolling(SPIKE_WINDOW, min_periods=SPIKE_MIN_PERIODS).mean())
rs = g['unit_price'].transform(lambda s: s.rolling(SPIKE_WINDOW, min_periods=SPIKE_MIN_PERIODS).std())
daraz_df['is_spike'] = (daraz_df['unit_price'] > rm + SPIKE_STD_MULT * rs).fillna(False).astype(int)

# companion flag, not a redefinition of is_spike: a spike is "discount-driven" if it
# coincides with a sale ending or an active discount shrinking (checked in EDA -- ~50%
# of raw spikes were explained this way, so is_spike alone conflates real volatility
# with predictable discount-cycle mechanics)
prev_discounted    = g['is_discounted'].shift(1)
prev_discount_pct  = g['discount_pct'].shift(1)
sale_ended         = (prev_discounted == True) & (daraz_df['is_discounted'] == False)
discount_shrinking = (prev_discounted == True) & (daraz_df['is_discounted'] == True) & (daraz_df['discount_pct'] < prev_discount_pct)
daraz_df['is_discount_driven_spike'] = (
    daraz_df['is_spike'].astype(bool) & (sale_ended.fillna(False) | discount_shrinking.fillna(False))
).astype(int)

print(f"{len(daraz_df)} rows, {daraz_df['is_spike'].sum()} spike-flagged "
      f"({daraz_df['is_discount_driven_spike'].sum()} discount-driven), {int(daraz_df['is_discounted'].sum())} discounted")


## 6. Day vs night check (exploratory)

In [ ]:
def day_night_stats(d):
    x = d.copy()
    x['run_period'] = np.where(x['hour'] < 12, 'morning', 'evening')
    stats = x.groupby(['category', 'run_period'])['unit_price'].agg(['count', 'mean', 'median', 'std']).round(2)
    print(stats)
    pivot = x.groupby(['category', 'run_period'])['unit_price'].median().unstack('run_period')
    if {'morning', 'evening'}.issubset(pivot.columns):
        pivot['pct_diff'] = ((pivot['evening'] - pivot['morning']) / pivot['morning'] * 100).round(2)
        print(pivot[['pct_diff']].sort_values('pct_diff', ascending=False))
    return stats

day_night_stats(daraz_df)


## 7. Paired same-product day vs night check

Isolates a real per-product effect from category-level session-composition
noise. Wilcoxon per category, `H0` = no systematic morning/evening
difference — ~6 tests, no multiple-comparisons correction, so treat one
isolated low p-value skeptically.

In [ ]:
def paired_check(d):
    x = d.copy()
    x['run_period'] = np.where(x['hour'] < 12, 'morning', 'evening')
    x['obs_date']   = x['scraped_at'].dt.date

    daily = x.groupby(['url', 'category', 'obs_date', 'run_period'])['unit_price'].median().reset_index()
    pivot = daily.pivot_table(index=['url', 'category', 'obs_date'], columns='run_period', values='unit_price')
    paired = pivot.dropna(subset=['morning', 'evening']).reset_index()
    paired['diff']     = paired['evening'] - paired['morning']
    paired['pct_diff'] = paired['diff'] / paired['morning'] * 100

    print(f"paired product-days: {len(paired)}")
    print(paired.groupby('category')['pct_diff'].agg(['count', 'mean', 'median', 'std']).round(3))

    for cat, grp in paired.groupby('category'):
        vals = grp['diff'].values
        vals = vals[vals != 0]
        if len(vals) < 10:
            print(f"  {cat}: n={len(vals)} -- too few to test")
            continue
        stat, p = wilcoxon(vals)
        print(f"  {cat}: n={len(vals)} p={p:.4f}" + ("  <-- possible effect" if p < 0.05 else ""))
    return paired

daraz_paired = paired_check(daraz_df)


## 8. Session-level export

In [ ]:
daraz_path = OUTDIR / 'forecasting_daraz_clean.csv'
daraz_df.to_csv(daraz_path, index=False)
print(f"{daraz_path} -- {len(daraz_df)} rows, {len(daraz_df.columns)} columns")


## 9. Collapse to daily (modeling grain)

95-100% of morning/evening pairs are identical (sections 6-7) — one row
per product per day from here on. `unit_price` takes the day's median;
discount/rating fields take the max across whichever session(s) exist that
day.

In [ ]:
d = daraz_df.copy()
d['obs_date'] = d['scraped_at'].dt.date

daraz_daily = d.groupby(['url', 'obs_date']).agg(
    category=('category', 'first'), source=('source', 'first'), name=('name', 'first'),
    unit_type=('unit_type', 'first'), unit_value=('unit_value', 'first'),
    unit_price=('unit_price', 'median'), final_price=('final_price', 'median'),
    original_price=('original_price', 'median'), sale_price=('sale_price', 'median'),
    is_discounted=('is_discounted', 'max'), discount_pct=('discount_pct', 'max'),
    rating=('rating', 'max'), review_count=('review_count', 'max'),
    sessions_that_day=('unit_price', 'size'),
).reset_index()

daraz_daily['scraped_at']  = pd.to_datetime(daraz_daily['obs_date'])
daraz_daily['day_of_week'] = daraz_daily['scraped_at'].dt.dayofweek
daraz_daily['week']        = daraz_daily['scraped_at'].dt.isocalendar().week.astype(int)
daraz_daily['month']       = daraz_daily['scraped_at'].dt.month
daraz_daily['is_weekend']  = daraz_daily['day_of_week'].isin([5, 6])
daraz_daily['has_rating']  = daraz_daily['rating'].notna()
daraz_daily['has_review']  = daraz_daily['review_count'].notna()
daraz_daily['rating_clean']       = daraz_daily['rating'].fillna(0)
daraz_daily['review_count_clean'] = daraz_daily['review_count'].fillna(0)

print(f"{len(d)} session rows -> {len(daraz_daily)} product-days "
      f"({(daraz_daily['sessions_that_day'] == 2).sum()} had both sessions)")


## 10. Re-engineer features on the daily grid

In [ ]:
daraz_daily = daraz_daily.sort_values(['url', 'scraped_at']).reset_index(drop=True)
g = daraz_daily.groupby('url', group_keys=False)

for lag in LAG_WINDOWS:
    daraz_daily[f'price_lag_{lag}'] = g['unit_price'].shift(lag)

daraz_daily['rolling_mean_7']  = g['unit_price'].transform(lambda s: s.rolling(ROLLING_SHORT, min_periods=1).mean())
daraz_daily['rolling_std_7']   = g['unit_price'].transform(lambda s: s.rolling(ROLLING_SHORT, min_periods=1).std())
daraz_daily['rolling_mean_14'] = g['unit_price'].transform(lambda s: s.rolling(ROLLING_LONG, min_periods=1).mean())
daraz_daily['rolling_min_7']   = g['unit_price'].transform(lambda s: s.rolling(ROLLING_SHORT, min_periods=1).min())
daraz_daily['rolling_max_7']   = g['unit_price'].transform(lambda s: s.rolling(ROLLING_SHORT, min_periods=1).max())

daraz_daily['price_change']     = g['unit_price'].diff(1)
daraz_daily['price_change_pct'] = g['unit_price'].transform(lambda s: s.pct_change(1) * 100)
daraz_daily['price_accel']      = g['price_change'].transform(lambda s: s.diff(1))

daraz_daily['discount_rate'] = daraz_daily['discount_pct'] / 100
daraz_daily['has_discount']  = daraz_daily['is_discounted'].astype(int)
daraz_daily['rolling_discount_rate'] = g['has_discount'].transform(lambda s: s.rolling(ROLLING_SHORT, min_periods=1).mean())

daraz_daily['log_review_count'] = np.log1p(daraz_daily['review_count_clean'])
daraz_daily['rating_weighted']  = daraz_daily['rating_clean'] * daraz_daily['has_review'].astype(int)

rm = g['unit_price'].transform(lambda s: s.rolling(SPIKE_WINDOW, min_periods=SPIKE_MIN_PERIODS).mean())
rs = g['unit_price'].transform(lambda s: s.rolling(SPIKE_WINDOW, min_periods=SPIKE_MIN_PERIODS).std())
daraz_daily['is_spike'] = (daraz_daily['unit_price'] > rm + SPIKE_STD_MULT * rs).fillna(False).astype(int)

prev_discounted    = g['is_discounted'].shift(1)
prev_discount_pct  = g['discount_pct'].shift(1)
sale_ended         = (prev_discounted == True) & (daraz_daily['is_discounted'] == False)
discount_shrinking = (prev_discounted == True) & (daraz_daily['is_discounted'] == True) & (daraz_daily['discount_pct'] < prev_discount_pct)
daraz_daily['is_discount_driven_spike'] = (
    daraz_daily['is_spike'].astype(bool) & (sale_ended.fillna(False) | discount_shrinking.fillna(False))
).astype(int)

print(f"{len(daraz_daily)} rows, {daraz_daily['is_spike'].sum()} spike-flagged "
      f"({daraz_daily['is_discount_driven_spike'].sum()} discount-driven)")


## 11. Export daily dataset

In [ ]:
daraz_daily_path = OUTDIR / 'forecasting_daraz_daily_clean.csv'
daraz_daily.to_csv(daraz_daily_path, index=False)
print(f"{daraz_daily_path} - {len(daraz_daily)} rows, {len(daraz_daily.columns)} columns")


## 12. Summary

In [ ]:
print(f"products.csv: {len(products)} | product_prices_recalculated.csv: {len(original_prices)}")
# print(f"Daraz session-level: {len(daraz_df)} rows, {daraz_df['url'].nunique()} products -> {daraz_path}")
print(f"Daraz daily         : {len(daraz_daily)} rows -> {daraz_daily_path}")


## Graph

In [ ]:
product_median_price = (
    daraz_daily.groupby(['url', 'category'])['unit_price'].median().reset_index()
)

categories = sorted(product_median_price['category'].unique())
fig, axes = plt.subplots(len(categories), 2, figsize=(11, 3 * len(categories)))

for i, cat in enumerate(categories):
    prices = product_median_price.loc[product_median_price['category'] == cat, 'unit_price']
    axes[i, 0].hist(prices, bins=30, color='steelblue')
    axes[i, 0].set_title(f'{cat} — raw price (n={len(prices)})')
    axes[i, 1].hist(np.log1p(prices), bins=30, color='darkorange')
    axes[i, 1].set_title(f'{cat} — log price')

plt.tight_layout()
plt.show()

stats = product_median_price.groupby('category')['unit_price'].agg(
    n='size', min='min',
    p10=lambda s: s.quantile(0.10), median='median', p90=lambda s: s.quantile(0.90),
    max='max',
).round(2)
stats['span_ratio'] = (stats['max'] / stats['min']).round(1)
print(stats)